# Ejercicio 9: Discusión

**Objetivo:** Reflexionar sobre las características de DuckDB, las ventajas y limitaciones del flujo de trabajo, y lo aprendido durante el laboratorio.

**Equipo:** Abby Donis (22440), Hansel Lopez (19026), Fabian Prado (23427)

## 9.1 Características de DuckDB más útiles

**1. Consulta directa sobre Parquet:** DuckDB puede consultar archivos Parquet sin importarlos a tabla, lo que elimina el tiempo de carga inicial y permite exploración rápida.

**2. Glob patterns en read_parquet:** La capacidad de usar `data/raw/*/*/*.parquet` permite consultar múltiples archivos y años automáticamente, sin listar cada archivo manualmente.

**3. union_by_name = true:** Cuando los archivos tienen columnas nuevas (como request_source en junio 2026), DuckDB las incluye automáticamente sin romper las consultas existentes.

**4. Funciones de metadatos:** `parquet_file_metadata()` permite verificar la integridad de los archivos sin leer los datos, útil para validación de descargas.

**5. Compatibilidad con pandas:** `con.sql().df()` convierte resultados directamente a DataFrame, facilitando la integración con el ecosistema Python.

## 9.2 Ventajas y limitaciones de consultar Parquet directamente

**Ventajas:**
- **Sin tiempo de carga:** Los datos se leen bajo demanda, ideal para exploración
- **Espacio eficiente:** No se duplican los datos en una base materializada
- **Incremental:** Archivos nuevos entran automáticamente sin reconfigurar
- **Flexibilidad:** Las consultas pueden cambiar sin costo de reimportación

**Limitaciones:**
- **Velocidad:** 16-20x más lento que tabla materializada en consultas repetidas
- **Sin índices:** Cada consulta escanea los archivos completos
- **Sin transacciones:** No hay rollback ni consistencia entre consultas
- **Memoria:** Consultas complejas pueden consumir mucha RAM

## 9.3 Ventajas y limitaciones de tablas materializadas en DuckDB

**Ventajas:**
- **Velocidad:** 16-20x más rápido que Parquet directo
- **Índices:** DuckDB crea índices automáticos para acelerar filtros
- **Optimización:** El motor optimiza el almacenamiento para consultas analíticas
- **Consistencia:** Los datos están en un estado conocido durante las consultas

**Limitaciones:**
- **Tiempo de carga:** 12-28 segundos para 30-70M filas
- **Espacio:** Duplica los datos (Parquet + tabla)
- **Rigidez:** Cambios en el esquema requieren recrear la tabla
- **Escritura única:** Solo un proceso puede escribir a la vez (limitación de DuckDB)

## 9.4 Ventajas frente a cargar todo con Pandas

**1. Memoria:** DuckDB puede consultar datos que no caben en RAM, Pandas no. Con 100M filas y 20 columnas, Pandas necesitaría ~16GB RAM, DuckDB funciona con 2GB.

**2. Velocidad:** DuckDB es 10-100x más rápido que Pandas en agregaciones porque usa columnar storage y vectorización.

**3. SQL:** Las consultas son más legibles y mantenibles que cadenas de métodos de Pandas.

**4. Concurrencia:** DuckDB maneja múltiples consultas simultáneas, Pandas no.

**5. Tipado:** DuckDB infiere tipos correctamente desde Parquet, Pandas a veces falla con fechas y categorías.

## 9.5 Características del sistema que permiten incorporar nuevos datos con cambios mínimos

**1. Vistas sobre glob patterns:** Las vistas en `sql/00_vistas.sql` usan `data/raw/*/*/*.parquet`, así que cualquier año nuevo que caiga en esa estructura entra automáticamente.

**2. Separación de datos y análisis:** Los datos viven en `data/raw/` y las consultas en `sql/`. Agregar datos no requiere cambiar código.

**3. Extracción de año desde el archivo:** La vista `viajes_validos` extrae el año del nombre del archivo con `split_part(filename, '/', -2)`, por lo que funciona con cualquier año.

**4. Script de descarga parametrizable:** El script acepta `--anios 2024 2025 2026` y descarga solo lo que falta, sin repetir archivos existentes.

**5. Consultas agnósticas al año:** Las consultas de análisis agrupan por `anio` y `mes`, así que pueden distinguir entre años sin modificación.

## 9.6 Qué parte del proceso debería automatizarse en producción

**1. Descarga de datos:** Un cron job que ejecute `download_data.py` diariamente para bajar los archivos nuevos automáticamente.

**2. Verificación de integridad:** El script `--verificar` debería correr después de cada descarga y alertar si hay archivos incompletos.

**3. Materialización de tablas:** Si el tablero se consulta frecuentemente, la tabla de DuckDB debería recrearse cada noche con los datos nuevos.

**4. Backups:** La base materializada debería respaldarse automáticamente para evitar perder el trabajo de materialización.

**5. Monitoreo:** Alertas si el volumen de viajes cae abruptamente (posible problema de datos) o si el tiempo de consulta se dispara (posible problema de rendimiento).

## 9.7 Decisiones de diseño importantes para mantener el proyecto reproducible

**1. Docker:** El `Dockerfile` fija las versiones de Python, DuckDB y dependencias, así cualquier persona levanta el mismo ambiente.

**2. Datos fuera de Git:** Los archivos Parquet no se versionan, solo los scripts que los descargan. Esto mantiene el repositorio ligero y los datos se obtienen de la fuente oficial.

**3. SQL separado de Python:** Las consultas viven en `sql/` y se ejecutan por nombre, así la consulta documentada y la ejecutada son la misma.

**4. Rutas relativas:** `scripts/consultas.py` resuelve rutas desde la raíz del laboratorio, así los cuadernos funcionan desde cualquier directorio.

**5. Script de descarga verificable:** El script compara lo descargado con lo publicado por el servidor, así se puede reproducir la descarga y verificar completitud.

**6. Notebooks numerados:** El orden de ejecución es explícito (01, 02, 03...), así cualquiera puede reproducir el análisis en el mismo orden.

## 9.8 Lo aprendido sobre manejo de datos que no sería evidente con conjuntos pequeños

**1. La calidad de datos importa más a escala:** Con 100M filas, problemas como horas nulas, montos negativos o fechas fuera de mes afectan miles de registros. En conjuntos pequeños estos problemas pasan desapercibidos.

**2. El costo de materialización se amortiza:** Crear una tabla tarda 12-28 segundos, pero si se consulta 100 veces, el ahorro de tiempo es enorme. Con conjuntos pequeños este tradeoff no es evidente.

**3. Los patrones temporales requieren múltiples años:** La estacionalidad y tendencias solo son visibles con datos de 2024, 2025 y 2026. Un solo año no muestra el panorama completo.

**4. La segmentación del mercado es estable:** La brecha entre amarillos y verdes se mantiene constante en los tres años, lo que sugiere que son mercados distintos que no convergen.

**5. El diseño incremental es esencial:** Agregar 2024 y 2025 sin tocar consultas solo fue posible porque el sistema fue diseñado para crecer. En conjuntos pequeños es tentador hacer soluciones ad hoc que no escalan.